In [1]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [ ]:
from typing import Literal, NotRequired, List, Dict
from typing_extensions import TypedDict
from dataclasses import dataclass

from langgraph.prebuilt.chat_agent_executor import AgentState

class Todo(TypedDict):
    content: str
    status: Literal['pending', 'in_progress', 'completed']


class ParallelizerAgent(AgentState):
    todos: NotRequired[List[Todo]]


@dataclass
class TokenUsage:
    input_tokens: int = 0
    output_tokens: int = 0
    total_token: int = 0

    def update(self, usage: Dict[str, int]):
        self.input_tokens = usage.get("prompt_tokens", 0)
        self.output_tokens = usage.get("completion_tokens", 0)
        self.total_tokens = usage.get("total_tokens", 0)

NameError: name 'Dict' is not defined

In [18]:
import time
import sys
import subprocess
import shutil
from pathlib import Path
from langchain_core.tools import tool, InjectedToolCallId
from typing import Literal, Annotated
from langgraph.prebuilt import InjectedState
from langgraph.types import Command
from langchain_core.messages import ToolMessage, AIMessage
from consts import WRITE_TODO_TOOL_DESC

BASE_DIR = Path("~/projects/llm-parallel-bench/llm_written").expanduser().resolve()
TIMEOUT_SEC = 20
MAX_OUTPUT_BYTES = 300_000


def _safe_path(filename: str) -> Path:
    """
    Resolve a path under BASE_DIR; strip any path traversal.
    Accepts either a bare filename or an absolute path already inside BASE_DIR.
    """
    p = Path(filename).expanduser()
    if not p.is_absolute():
        p = (BASE_DIR / p.name).resolve()
    else:
        p = p.resolve()
    if p == BASE_DIR:
        raise FileNotFoundError("Empty or invalid filename.")
    if BASE_DIR not in p.parents:
        raise PermissionError("Path escapes workspace.")
    return p


def _truncate(s: str, limit: int = MAX_OUTPUT_BYTES) -> str:
    if s is None:
        return ""
    b = s.encode("utf-8", errors="replace")
    if len(b) <= limit:
        return s
    head = limit // 2
    tail = limit - head - len("\n...\n".encode())
    return (b[:head] + b"\n...\n" + b[-tail:]).decode("utf-8", errors="replace")


@tool
def read_file(filename: str):
    """
    Read content of a file

    Args:
        - filename: str filename
    Returns: 
        - status: successful | error
        - content: str content of the file
        - stderr(optional): str error
    """

    try:
        path = _safe_path(filename)
        if not path.exists():
            return {"status": "error", "content": "", "stderr": "File does not exist"}
        elif path.is_dir():
            return {"status": "error", "content": "", "stderr": "File is a directory."}

        with open(path, "r") as file:
            content = file.read()

        return {"status": "successful", "content": content, "stderr": ""}
    except PermissionError as e:
        return {"status": "error", "path": str(path), "stderr": f"Permission error: {e}"}
    except Exception as e:
        return {"status": "error", "path": str(path), "stderr": f"Unexpected error: {e}"}


@tool
def write_file(filename: str, code: str):
    """
    Write content into a file and save it.

    Args:
        filename: filename of the file with extension. eg: linearsearch.py, bfs.go
        content: content to be written to the file.
    """
    try:
        safe_name = Path(filename).name
        if not safe_name:
            return {"status": "error", "error": "Empty filename"}
        path = (BASE_DIR / safe_name).resolve()

        if BASE_DIR not in path.parents and path != BASE_DIR / safe_name:
            return {"status": "error", "error": "Invalid path"}

        path.parent.mkdir(parents=True, exist_ok=True)

        with path.open("w", encoding="utf-8", newline="\n") as f:
            n = f.write(code)

        return {"status": "successful", "bytes_written": n, "path": str(path)}
    
    except Exception as e:
        return {"status": "error", "error": str(e)}
    

@tool
def run_code(filename: str, language: Literal['python', 'go', 'cpp']):
    """
    Run a program and capture stdout/stderr.

    Args: 
        filename: Program entry file or binary.
        language: One of 'python' | 'go' | 'cpp'.

    Returns: dict with keys:
        - status: "successful" | "error"
        - returncode: int | None
        - stdout: str (truncated if large)
        - stderr: str (truncated if large)
        - duration_sec: float
        - cmd: list[str] (actual command executed)
        - path: str (resolved path used)
        - note: str (optional details)
    """
    print("run_code", filename)
    start = time.monotonic()
    try:
        path = _safe_path(filename)

        cmd = [sys.executable, str(path)]

        proc = subprocess.run(
            cmd,
            cwd=str(BASE_DIR),
            capture_output=True,
            text=True,
            timeout=TIMEOUT_SEC
        )

        duration = round(time.monotonic() - start, 6)

        return {
            "status": "successful" if proc.returncode == 0 else "error",
            "returncode": proc.returncode,
            "stdout": _truncate(proc.stdout),
            "stderr": _truncate(proc.stderr),
            "duration_sec": duration,
            "cmd": cmd,
            "path": str(path),
        }
    except subprocess.TimeoutExpired as e:
        duration = round(time.monotonic() - start, 6)
        out = e.stdout.decode("utf-8", errors="replace") if isinstance(e.stdout, (bytes, bytearray)) else (e.stdout or "")
        err = e.stderr.decode("utf-8", errors="replace") if isinstance(e.stderr, (bytes, bytearray)) else (e.stderr or "")

        return {
            "status": "error",
            "returncode": None,
            "stdout": _truncate(out),
            "stderr": _truncate(err) + "\n[Timeout]",
            "duration_sec": duration,
            "cmd": getattr(e, "cmd", []),
            "path": filename,
        }
    
    except FileNotFoundError as e:
        duration = round(time.monotonic() - start, 6)
        return {
            "status": "error",
            "returncode": None,
            "stdout": "",
            "stderr": str(e),
            "duration_sec": duration,
            "cmd": [],
            "path": filename,
        }
    except PermissionError as e:
        duration = round(time.monotonic() - start, 6)
        return {
            "status": "error",
            "returncode": None,
            "stdout": "",
            "stderr": f"Permission error: {e}",
            "duration_sec": duration,
            "cmd": [],
            "path": filename,
        }
    except Exception as e:
        duration = round(time.monotonic() - start, 6)
        return {
            "status": "error",
            "returncode": None,
            "stdout": "",
            "stderr": f"Unexpected error: {e}",
            "duration_sec": duration,
            "cmd": [],
            "path": filename,
        }

@tool
def compile_code(filename: str):
    """
    Compile C++ program into binary file.

    Args:
        filename: filename of the c++ file.
    
    Returns: dict with keys:
        - status: "successful" | "error"
        - returncode: int | None
        - stdout: str (truncated if large)
        - stderr: str (truncated if large)
        - duration_sec: float
        - cmd: list[str] (actual command executed)
        - path: str (resolved path used)
    """

    start = time.monotonic()
    try:
        path = _safe_path(filename)
        if path.suffix.lower() not in {"cpp", ".cc", ".cxx", 'c++'}:
            return {
                "status": "error",
                "returncode": None,
                "stdout": "",
                "stderr": "Expected a C++ source file (.cpp).",
                "duration_sec": round(time.monotonic() - start, 6),
                "cmd": [],
                "path": str(path),
            } 
        
        out = path.with_suffix("")
        if sys.platform.startswith("win"):
            out = out.with_suffix("exe")
        
        cxx = shutil.which("g++")
        base_cmd = [cxx, "-std=c++17", "-O3", str(src), "-o", str(out)]
        omp_cmd  = [cxx, "-std=c++17", "-O3", "-fopenmp", str(src), "-o", str(out)]

        for cmd in (omp_cmd, base_cmd):
            try:
                proc = subprocess.run(
                    cmd,
                    cwd=str(BASE_DIR),
                    capture_output=True,
                    text=True,
                    timeout=TIMEOUT_SEC,
                )
                duration = round(time.monotonic() - start, 6)
                if proc.returncode == 0:
                    return {
                        "status": "successful",
                        "returncode": 0,
                        "stdout": _truncate(proc.stdout),
                        "stderr": _truncate(proc.stderr),
                        "duration_sec": duration,
                        "cmd": cmd,
                        "path": str(path),
                    }
                last_result = {
                    "status": "error",
                    "returncode": proc.returncode,
                    "stdout": _truncate(proc.stdout),
                    "stderr": _truncate(proc.stderr),
                    "duration_sec": duration,
                    "cmd": cmd,
                    "path": str(path),
                }
            except subprocess.TimeoutExpired as e:
                duration = round(time.monotonic() - start, 6)
                out_s = e.stdout.decode("utf-8", errors="replace") if isinstance(e.stdout, (bytes, bytearray)) else (e.stdout or "")
                err_s = e.stderr.decode("utf-8", errors="replace") if isinstance(e.stderr, (bytes, bytearray)) else (e.stderr or "")
                return {
                    "status": "error",
                    "returncode": None,
                    "stdout": _truncate(out_s),
                    "stderr": _truncate(err_s) + "\n[Timeout]",
                    "duration_sec": duration,
                    "cmd": getattr(e, "cmd", cmd),
                    "path": str(path),
                }
        
        return last_result
    except FileNotFoundError as e:
        return {
            "status": "error",
            "returncode": None,
            "stdout": "",
            "stderr": str(e),
            "duration_sec": round(time.monotonic() - start, 6),
            "cmd": [],
            "path": filename,
        }
    except PermissionError as e:
        return {
            "status": "error",
            "returncode": None,
            "stdout": "",
            "stderr": f"Permission error: {e}",
            "duration_sec": round(time.monotonic() - start, 6),
            "cmd": [],
            "path": filename,
        }
    except Exception as e:
        return {
            "status": "error",
            "returncode": None,
            "stdout": "",
            "stderr": f"Unexpected error: {e}",
            "duration_sec": round(time.monotonic() - start, 6),
            "cmd": [],
            "path": filename,
        }


@tool(description=WRITE_TODO_TOOL_DESC)
def  write_todos(todos: list[Todo], tool_call_id: Annotated[str, InjectedToolCallId]):
    """
    Create or update the agent's TODO list for task planning and tracking.

    Args:
        todos: List of Todo items with content and status
        tool_call_id: Tool call identifier for message response

    Returns:
        Command to update agent state with new TODO list
    """

    return Command(
        update={
            "todos": todos,
            "messages": [
                ToolMessage(f"Updated TODO list to {todos}.", tool_call_id=tool_call_id)
            ]
        }
    )

@tool(parse_docstring=True)
def read_todos(state: Annotated[ParallelizerAgent, InjectedState], tool_call_id: Annotated[str, InjectedToolCallId]):
    """Read the current TODO list from the agent state.

    This tool allows the agent to retrieve and review the current TODO list
    to stay focused on remaining tasks and track progress through complex workflows.

    Args:
        state: Injected agent state containing the current TODO list
        tool_call_id: Injected tool call identifier for message tracking

    Returns:
        Formatted string representation of the current TODO list
    """
    todos = state.get("todos", [])
    if not todos:
        return "No todos currently in the list."
    
    result = "Current TODO list:\n"
    status_emoji = {"pending": "⏳", "in_progress": "🔄", "completed": "✅"}
    for idx, todo in enumerate(todos, start=1):
        emoji = status_emoji.get(todo["status"], "❓")
        result += f"{idx}. {emoji} {todo['content']} ({todo['status']})\n"
    
    return result.strip()

@tool(parse_docstring=True)
def ls():
    """
    List all files in the working directory.
    """
    file_list = [p.name for p in BASE_DIR.iterdir() if p.is_file()]
    if not file_list:
        return "No files found in the working directory."
    
    result = "Current files in working directory:\n"
    for fname in file_list:
        result += f"- {fname}\n"
    
    return result.strip()


@tool(parse_docstring=True)
def think_tool(reflection: str) -> str:
    """Tool for strategic reflection on research progress and decision-making.

    Use this tool after each search to analyze results and plan next steps systematically.
    This creates a deliberate pause in the research workflow for quality decision-making.

    When to use:
    - After receiving search results: What key information did I find?
    - Before deciding next steps: Do I have enough to answer comprehensively?
    - When assessing research gaps: What specific information am I still missing?
    - Before concluding research: Can I provide a complete answer now?
    - How complex is the question: Have I reached the number of search limits?

    Reflection should address:
    1. Analysis of current findings - What concrete information have I gathered?
    2. Gap assessment - What crucial information is still missing?
    3. Quality evaluation - Do I have sufficient evidence/examples for a good answer?
    4. Strategic decision - Should I continue searching or provide my answer?

    Args:
        reflection: Your detailed reflection on research progress, findings, gaps, and next steps

    Returns:
        Confirmation that reflection was recorded for decision-making
    """
    return f"Reflection recorded: {reflection}"


In [19]:
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from consts import MONO_AGENT

agent = create_react_agent(
    model=ChatOpenAI(model="gpt-4o", temperature=0),  
    prompt=MONO_AGENT,  
    tools=[read_file, write_file, ls, write_todos, read_todos, run_code, think_tool],
    state_schema=ParallelizerAgent
)

In [ ]:
EX_REQUEST = """
from typing import List, Dict

class Graph:
    def __init__(self) -> None:
        self.vertices: Dict[int, list[int]] = {}
    
    def add_edge(self, from_vertex: int, to_vertex: int) -> None:
        if from_vertex not in self.vertices:
            self.vertices[from_vertex] = []
        if to_vertex not in self.vertices:
            self.vertices[to_vertex] = []
        self.vertices[from_vertex].append(to_vertex)
        self.vertices[to_vertex].append(from_vertex)


def bfs(graph: Graph, start_vertex: int) -> list[int]:
    if start_vertex not in graph.vertices:
        return []
    
    visited = set()
    result = []
    queue = [start_vertex]
    while queue:
        current = queue.pop(0)
        if current not in visited:
            visited.add(current)
            result.append(current)
            for neighbor in graph.vertices.get(current, []):
                if neighbor not in visited:
                    queue.append(neighbor)

    return result
"""

token_usage = TokenUsage()
config = {"recursion_limit": 50}
inputs = {"messages": [("user", f"Parallelize the following \n\n{EX_REQUEST}")]}

i = 1

for event in agent.stream(inputs, config=config, stream_usage=True):
    for k, v in event.items():
        print(f"\n=== Step {i+1} ({k}) ===")
        for message in v["messages"]:
            if isinstance(message, AIMessage):
                token_usage.update(message.response_metadata["token_usage"])
                
                if message.tool_calls:  
                    for tool_call in message.tool_calls:
                        print(f"[Tool call] {tool_call['name']}({tool_call['args']})")
                else:
                    print(f"[Message] {message.content}")
            elif isinstance(message, ToolMessage):
                print(f"[Tool message] {message.content}")


if final_state:
    output_msg = final_state["data"]["output"]
    usage = output_msg.response_metadata.get("token_usage", {})

    print("--- Token Usage ---")
    print("Prompt tokens:", usage.get("prompt_tokens"))
    print("Completion tokens:", usage.get("completion_tokens"))
    print("Total tokens:", usage.get("total_tokens"))
    


=== Step 2 ===
[Tool call] write_todos({'todos': [{'content': 'Plan parallelization strategy for BFS algorithm.', 'status': 'pending'}, {'content': 'Capture baseline performance of the sequential BFS implementation.', 'status': 'pending'}, {'content': 'Implement parallel BFS using concurrent data structures.', 'status': 'pending'}, {'content': 'Create differential tests to compare sequential and parallel BFS outputs.', 'status': 'pending'}, {'content': 'Run differential tests and performance checks for BFS.', 'status': 'pending'}, {'content': 'Refine implementation if needed based on test results.', 'status': 'pending'}, {'content': 'Finalize the parallel BFS implementation and documentation.', 'status': 'pending'}]})
[Tool call] ls({})
{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Lu5AvkedmSOU7rL81DfiLmIf', 'function': {'arguments': '{"todos": [{"content": "Plan parallelization strategy for BFS algorithm.", "status": "pending"}, {"conten

In [16]:
event["agent"]["messages"][0].response_metadata["token_usage"]

{'completion_tokens': 148,
 'prompt_tokens': 2689,
 'total_tokens': 2837,
 'completion_tokens_details': {'accepted_prediction_tokens': 0,
  'audio_tokens': 0,
  'reasoning_tokens': 0,
  'rejected_prediction_tokens': 0},
 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1920}}